# Import Materials
This example shows how to import materials.

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import tempfile
import time
from pathlib import Path

import toml
from pyedb import Edb


Define constants

In [2]:
AEDT_VERSION = "2026.1"

## Preparation

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = Path(temp_folder.name) / "test.aedb"
edbapp = Edb(edbpath=str(file_edb), version=AEDT_VERSION)

PyEDB INFO: gRPC mode enabled.


PyEDB INFO: Logger is initialized in EDB.


PyEDB INFO: legacy v0.88.dev0


PyEDB INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


WARNING - The installed EDB_RPC_Server does not support shared-memory IPC. Falling back to standard gRPC communication. Update to the latest version of AEDT for improved performance.


PyEDB INFO: Grpc session started: pid=3560


PyEDB INFO: RPC session acquired (open databases: 1)


PyEDB INFO: Refreshing the Components dictionary.


PyEDB INFO: EDB C:\Users\ansys\AppData\Local\Temp\tmplkal45u2.ansys\test.aedb created correctly.


PyEDB INFO: EDB initialized.


## Create configure file

## Add distributed ports

Keywords

- **name**. Name of the material.
- **permittivity**.
- **conductivity**.
- **dielectric_loss_tangent**.
- **magnetic_loss_tangent**.
- **mass_density**.
- **permeability**.
- **poisson_ratio**.
- **specific_heat**.
- **thermal_conductivity**.
- **thermal_modifier**.
  - **property_name**.
  - **basic_quadratic_c1**. The C1 value in the quadratic model.
  - **basic_quadratic_c2**. The C2 value in the quadratic model.
  - **basic_quadratic_temperature_reference**. The TempRef value in the quadratic model.
  - **advanced_quadratic_lower_limit**. The lower temperature limit where the quadratic model is valid.
  - **advanced_quadratic_upper_limit**. The upper temperature limit where the quadratic model is valid.
  - **advanced_quadratic_auto_calculate**. The flag indicating whether or not the LowerConstantThermalModifierVal and UpperConstantThermalModifierVal values should be auto calculated.
  - **advanced_quadratic_lower_constant**. The constant thermal modifier value for temperatures lower than LowerConstantThermalModifierVal
  - **advanced_quadratic_upper_constant**. The constant thermal modifier value for temperatures greater than UpperConstantThermalModifierVal.

In [4]:
materials = [
    {
        "name": "copper",
        "conductivity": 570000000,
        "thermal_modifier": [
            {
                "property_name": "conductivity",
                "basic_quadratic_c1": 0,
                "basic_quadratic_c2": 0,
                "basic_quadratic_temperature_reference": 22,
                "advanced_quadratic_lower_limit": -273.15,
                "advanced_quadratic_upper_limit": 1000,
                "advanced_quadratic_auto_calculate": True,
                "advanced_quadratic_lower_constant": 1,
                "advanced_quadratic_upper_constant": 1,
            },
        ],
    },
]
cfg = {"stackup": {"materials": materials}}
file_json = Path(temp_folder.name) / "edb_configuration.json"
with open(file_json, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Equivalent toml file looks like below

In [5]:
toml_string = toml.dumps(cfg)
print(toml_string)

[stackup]
[[stackup.materials]]
name = "copper"
conductivity = 570000000
[[stackup.materials.thermal_modifier]]
property_name = "conductivity"
basic_quadratic_c1 = 0
basic_quadratic_c2 = 0
basic_quadratic_temperature_reference = 22
advanced_quadratic_lower_limit = -273.15
advanced_quadratic_upper_limit = 1000
advanced_quadratic_auto_calculate = true
advanced_quadratic_lower_constant = 1
advanced_quadratic_upper_constant = 1





## Import configuration into example layout

In [6]:
edbapp.configuration.load(config_file=file_json)
edbapp.configuration.run()

PyEDB INFO: Updating nets finished. Time lapse 0:00:00.017138


PyEDB INFO: Updating components finished. Time lapse 0:00:00


PyEDB INFO: Creating pin groups finished. Time lapse 0:00:00


PyEDB INFO: Placing sources finished. Time lapse 0:00:00


PyEDB INFO: Applying materials finished. Time lapse 0:00:00.015741


PyEDB INFO: Applying padstack definitions and instances completed in 0.0050 seconds.


... (9 outputs left out)

PyEDB INFO: Caching finished.


PyEDB INFO: Applying setups completed in 0.0000 seconds.


PyEDB INFO: Applying operations completed in 0.0000 seconds.


True

## Save and close Edb
The temporary folder will be deleted once the execution of this script is finished. Replace **edbapp.save()** with
**edbapp.save_as("C:/example.aedb")** to keep the example project.

In [7]:
edbapp.save()
edbapp.close()

PyEDB INFO: RPC session released (open databases: 0)


True

In [8]:
# Wait 3 seconds before cleaning the temporary directory.
time.sleep(3)

### Clean up

All project files are saved in the folder ``temp_folder.name``.
If you've run this example as a Jupyter notebook, you
can retrieve those project files. The following cell
removes all temporary files, including the project folder.

In [9]:
temp_folder.cleanup()